# 4 - Compare models and choose an operating point

The playground. Load several checkpoints, run them on the same frames, and
work out which one to ship and at what confidence threshold.

Nothing here trains anything. It is for the question that comes after
training: *given these models, what do we actually deploy?*

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dmd
import matplotlib.pyplot as plt
from dmd import DMD_CLASSES, load_coco, load_model, predict_images

HERE = Path.cwd().parent
DATA = Path.home() / "aerial-data"

## Pick the models and the frames

Point `MODELS` at any checkpoints you have. `None` means stock
DroneMegaDetector. They all share the same three classes in the same order,
so they are directly comparable.

In [ ]:
MODELS = {
    "stock DMD": None,
    "seals-v1": HERE / "runs" / "seals-v1" / "checkpoint_best_total.pth",
    "birds-v1": HERE / "runs" / "birds-v1" / "checkpoint_best_total.pth",
}
MODELS = {name: path for name, path in MODELS.items() if path is None or Path(path).exists()}
print("comparing:", list(MODELS))

EVAL = DATA / "noaa_eval" / "_annotations.coco.json"
truth = load_coco(EVAL)
image_ids = sorted(truth.images)
paths = [truth.image_path(i) for i in image_ids]
print(f"{len(paths)} images, {truth.box_count} boxes, {truth.per_class_counts()}")

## Run them all once

Predictions are cached in memory at a permissive threshold, so every
experiment below is instant - no re-running inference per setting.

In [ ]:
predictions = {}
for name, weights in MODELS.items():
    print(f"\n=== {name} ===")
    predictions[name] = predict_images(load_model(weights), paths, threshold=0.05)
print("\ndone")

## All of them on one chart

In [ ]:
figure, axis = plt.subplots(figsize=(8, 5))
sweeps = {}
for name in MODELS:
    rows = dmd.sweep(
        [(predictions[name][truth.image_path(i)], truth.boxes.get(i, [])) for i in image_ids],
        iou_threshold=0.5,
    )
    sweeps[name] = rows
    axis.plot([r["confidence"] for r in rows], [r["f1"] for r in rows], "-o",
              markersize=4, label=name)
axis.set_xlabel("confidence threshold")
axis.set_ylabel("F1")
axis.set_ylim(0, 1.02)
axis.grid(alpha=0.3)
axis.legend()
axis.set_title("all models, all classes")
plt.show()

print("  model             conf    P       R       F1      blind")
for name, rows in sweeps.items():
    b = dmd.best_row(rows)
    print(f"  {name:17s} {b['confidence']:<7.2f} {b['precision']:<7.3f} "
          f"{b['recall']:<7.3f} {b['f1']:<7.3f} {b['blind_images']}")

## Per class, per model

A specialised model should win on its class without losing the others. This
is where you see whether it did.

In [ ]:
figure, axes = plt.subplots(
    1, len(DMD_CLASSES), figsize=(5 * len(DMD_CLASSES), 4), squeeze=False
)
for column, (class_id, class_name) in enumerate(enumerate(DMD_CLASSES)):
    axis = axes[0][column]
    count = sum(1 for boxes in truth.boxes.values() for b in boxes if b.class_id == class_id)
    for name in MODELS:
        rows = dmd.sweep(
            [
                (
                    [b for b in predictions[name][truth.image_path(i)]
                     if b.class_id == class_id],
                    [b for b in truth.boxes.get(i, []) if b.class_id == class_id],
                )
                for i in image_ids
            ],
            iou_threshold=0.5,
        )
        axis.plot([r["confidence"] for r in rows], [r["f1"] for r in rows], "-o",
                  markersize=3, label=name)
    axis.set_title(f"{class_name} ({count} boxes)", fontsize=10)
    axis.set_xlabel("confidence")
    axis.set_ylim(0, 1.02)
    axis.grid(alpha=0.3)
    if column == 0:
        axis.set_ylabel("F1")
        axis.legend(fontsize=8)
plt.show()

## Choosing a threshold: F1 is the wrong objective

F1 weighs a miss and a false alarm equally. For dispatch they are not equal.

A missed stranded animal is a welfare failure. A false alarm costs a
coordinator ten seconds of looking at a frame - and a human reviews every
detection anyway, which is the whole point of the design. So the cost of a
miss is far higher than the cost of a false positive.

Set `MISS_COST` to how many false alarms you would accept to avoid one miss,
and this picks the threshold that minimises total cost. Try 5, then 20, and
watch the recommendation move.

In [ ]:
MISS_COST = 10.0          # false alarms you would trade for one avoided miss

print(f"  cost = {MISS_COST} x misses + 1 x false alarms\n")
print("  model             conf    cost     P       R       missed  false")
recommended = {}
for name, rows in sweeps.items():
    best = min(rows, key=lambda r: MISS_COST * r["fn"] + r["fp"])
    recommended[name] = best
    cost = MISS_COST * best["fn"] + best["fp"]
    print(f"  {name:17s} {best['confidence']:<7.2f} {cost:<8.0f} {best['precision']:<7.3f} "
          f"{best['recall']:<7.3f} {best['fn']:<7d} {best['fp']}")

print("\n  For comparison, the best-F1 thresholds were:")
for name, rows in sweeps.items():
    print(f"    {name:17s} {dmd.best_row(rows)['confidence']:.2f}")
print("\n  The cost-weighted threshold is usually lower. That is correct:")
print("  it is buying recall with false alarms a human will filter.")

## Interactive: drag the threshold and watch the boxes

Needs `ipywidgets`. If the slider does not render, set `THRESHOLD` by hand in
the cell below it.

In [ ]:
try:
    from ipywidgets import Dropdown, FloatSlider, IntSlider, interact

    def preview(model_name, threshold, index):
        i = image_ids[index]
        path = truth.image_path(i)
        found = [b for b in predictions[model_name][path] if b.score >= threshold]
        image = dmd.draw_boxes(path, predictions[model_name][path],
                               truth.boxes.get(i, []), threshold=threshold)
        figure, axis = plt.subplots(figsize=(11, 7))
        axis.imshow(image)
        axis.axis("off")
        axis.set_title(
            f"{path.name} - {model_name} @ {threshold:.2f} - "
            f"{len(found)} detections, {len(truth.boxes.get(i, []))} labelled",
            fontsize=10,
        )
        plt.show()

    interact(
        preview,
        model_name=Dropdown(options=list(MODELS), description="model"),
        threshold=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.3, description="conf"),
        index=IntSlider(min=0, max=len(image_ids) - 1, value=0, description="frame"),
    )
except ImportError:
    print("ipywidgets not installed: pip install ipywidgets")

In [ ]:
# Static fallback, and the cell to use for screenshots.
MODEL_NAME = list(MODELS)[0]
THRESHOLD = 0.30

sample = image_ids[:6]
dmd.show_grid(
    [dmd.draw_boxes(truth.image_path(i), predictions[MODEL_NAME][truth.image_path(i)],
                    truth.boxes.get(i, []), threshold=THRESHOLD) for i in sample],
    columns=3,
    titles=[truth.image_path(i).name for i in sample],
)
plt.show()

## Does tiling help?

DroneMegaDetector's own batch script slices large images into 1280x1280
tiles with 100 px overlap and merges detections with NMS. It helps when
targets are small relative to the frame, because each tile gets upscaled to
the model's input resolution - and costs roughly one forward pass per tile.

Worth testing if your pixels-on-target histogram sits low.

In [ ]:
from PIL import Image

print(f"tile size {dmd.DMD_TILE_SIZE}, overlap {dmd.DMD_TILE_OVERLAP}")
print(f"model native resolution {dmd.DMD_NATIVE_RESOLUTION}, "
      f"patch size {dmd.DMD_PATCH_SIZE}")

widths = []
for path in paths[:20]:
    with Image.open(path) as handle:
        widths.append(handle.size)
print("frame sizes seen:", sorted(set(widths)))

if widths and max(w for w, _ in widths) <= dmd.DMD_TILE_SIZE:
    print("\nFrames are no larger than one tile, so tiling would change nothing here.")
    print("It matters for full-resolution stills, not for 720p/1080p video frames.")
else:
    print("\nFrames exceed one tile. Tiling is worth measuring:")
    print("  python -m pip show rfdetr   # then use DroneMegaDetector's")
    print("  inference-batch.py --tiled, and score its JSON with evaluate.py")

## Where are the remaining misses, across every model?

Frames that *no* model finds anything in are the most informative images you
have. They are what to label more of, or what tells you the limit is
physical rather than statistical.

In [ ]:
THRESHOLD_FOR_MISSES = 0.3

hard = []
for i in image_ids:
    gt = truth.boxes.get(i, [])
    if not gt:
        continue
    if all(
        not [b for b in predictions[name][truth.image_path(i)]
             if b.score >= THRESHOLD_FOR_MISSES]
        for name in MODELS
    ):
        hard.append(i)

print(f"{len(hard)} frames have ground truth that no model detects at {THRESHOLD_FOR_MISSES}")
if hard:
    sizes = [b.longest_side for i in hard for b in truth.boxes.get(i, [])]
    print(f"their boxes: median {sorted(sizes)[len(sizes) // 2]:.0f} px, "
          f"smallest {min(sizes):.0f}, largest {max(sizes):.0f}")
    print("Compare that against the ~32 px floor. If these are simply tiny,")
    print("more training data will not fix them - altitude or resolution will.")
    dmd.show_grid(
        [dmd.draw_boxes(truth.image_path(i), None, truth.boxes.get(i, [])) for i in hard[:6]],
        columns=3,
        titles=[truth.image_path(i).name for i in hard[:6]],
    )
    plt.show()

## Write down what you chose

The operating point is a decision, not a number that falls out of the data.
Record it with its reasoning so the next person - or you in three months -
knows why it is what it is.

In [ ]:
CHOSEN_MODEL = list(MODELS)[0]
CHOSEN_THRESHOLD = recommended.get(CHOSEN_MODEL, {}).get("confidence", 0.3)

row = next(r for r in sweeps[CHOSEN_MODEL] if r["confidence"] == CHOSEN_THRESHOLD)
dmd.save_json(
    {
        "chosen_model": CHOSEN_MODEL,
        "chosen_threshold": CHOSEN_THRESHOLD,
        "miss_cost_assumed": MISS_COST,
        "evaluated_on": str(EVAL),
        "images": len(paths),
        "metrics_at_operating_point": row,
        "all_models": {name: dmd.best_row(rows) for name, rows in sweeps.items()},
        "rationale": (
            "Threshold minimises MISS_COST x misses + false alarms, not F1. "
            "A missed stranded animal is a welfare failure; a false alarm costs "
            "a coordinator seconds, and a human reviews every detection anyway."
        ),
    },
    dmd.RESULTS / "operating_point.json",
)
print(f"\n{CHOSEN_MODEL} at {CHOSEN_THRESHOLD:.2f}: "
      f"recall {row['recall']:.3f}, precision {row['precision']:.3f}, "
      f"{row['fn']} missed, {row['fp']} false alarms")

## Things worth trying from here

- **Raise `MISS_COST`** and see how far recall goes before precision becomes
  unworkable for a reviewer.
- **Lower the IoU threshold** from 0.5 to 0.3 in `dmd.sweep`. If scores jump,
  the model is finding the animals but boxing them loosely - which for "is
  there something here?" may be fine.
- **Evaluate on one session at a time** to find out whether a site, a light
  condition or an altitude is driving the errors. That is usually more
  actionable than the aggregate.
- **Check the disagreements between models.** Frames where the seal model
  fires and stock DMD does not are what the fine-tune actually bought.